# 03 — Daily F10.7 Wavelet Analysis

This notebook demonstrates the daily solar-index wavelet workflow. It preserves the original F10.7 values (no mean subtraction), uses a Morlet wavelet with \(\omega_0=6\), and plots the period in days.

It expects the local `wave_python/waveletFunctions.py` implementation. The transform's 95% significance calculation and COI should be checked against that implementation and its documented conventions.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
WAVELET_DIR = PROJECT_ROOT / "wave_python"
CSV = PROJECT_ROOT / "data" / "raw" / "f107" / "F107_daily_1979_2010.csv"
if not CSV.exists():
    raise FileNotFoundError(CSV)
sys.path.insert(0, str(WAVELET_DIR))
from waveletFunctions import wavelet, wave_signif


In [ ]:
df = pd.read_csv(CSV)
df.columns = [str(c).strip().lower() for c in df.columns]
if not {"date", "f107"}.issubset(df.columns):
    raise ValueError(f"Expected date and f107 columns; got {df.columns.tolist()}")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["f107"] = pd.to_numeric(df["f107"], errors="coerce")
df = df.dropna(subset=["date", "f107"]).sort_values("date").drop_duplicates("date")
df = df.loc[df["f107"] > 0]
daily = df.set_index("date")["f107"].asfreq("D")
missing = int(daily.isna().sum())
print("Daily samples:", len(daily), "Missing values:", missing)
if missing:
    print("Review missing days before running the transform. This notebook intentionally does not silently interpolate them.")


In [ ]:
# Run only when the daily series is complete, or after you have made and documented a missing-data decision.
if daily.isna().any():
    raise ValueError("Missing daily values detected. Resolve and document them before wavelet analysis.")

f107 = daily.to_numpy(dtype=float)  # intentionally no mean subtraction
dt, pad, dj, s0 = 1.0, 1, 0.25, 2.0
mother, param = "MORLET", 6
j1 = int(np.floor(np.log2(len(f107) * dt / s0) / dj))
wave, period, scale, coi = wavelet(f107, dt, pad, dj, s0, j1, mother, param)
power = np.abs(wave) ** 2
lag1 = np.corrcoef(f107[:-1], f107[1:])[0, 1]
signif = np.asarray(wave_signif(f107, dt=dt, scale=scale, sigtest=0, lag1=lag1, siglvl=0.95, dof=-1, mother=mother, param=param)).squeeze()
print("Transform shape:", wave.shape)
print("Period range (days):", period.min(), period.max())
print("Lag-1 autocorrelation:", lag1)


In [ ]:
# Plot log10 power. Significance is calculated on original power.
log_power = np.log10(np.maximum(power, np.finfo(float).tiny))
x = np.arange(len(f107))
period_mask = (period >= 2) & (period <= 2000)
fig, ax = plt.subplots(figsize=(14, 6))
cf = ax.contourf(x, period[period_mask], log_power[period_mask], levels=50, cmap="turbo", extend="both")
ratio = power / np.asarray(signif).reshape(-1, 1)
ax.contour(x, period[period_mask], ratio[period_mask], levels=[1], colors="black", linewidths=1)
ax.plot(x, coi, "w--", linewidth=1.5, label="COI")
ax.set_yscale("log")
ax.set_ylim(2, 2000)  # short periods bottom, long periods top
ax.set_xlabel("Daily sample index")
ax.set_ylabel("Period (days)")
ax.set_title("F10.7 wavelet power (daily; no mean subtraction)")
fig.colorbar(cf, ax=ax, label="log10(Wavelet power)")
ax.legend()
fig.tight_layout()
plt.show()


## Interpretation cautions

The black contour indicates power/significance above 1 only if the library returns a per-scale significance threshold compatible with that ratio. The COI identifies edge-affected regions; peaks near or outside the reliable region require caution. Inspect the library's COI units and significance output shape before relying on the figure.
